# 🔧 Feature Engineering — Energy Grid Outage Prediction

Aggregates grid sensor readings per substation per day and joins with
outage events to create features for outage prediction.

**Reads from:** Lakehouse tables (ingested from CSV)

**Writes to:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, stddev, count,
    sum as spark_sum, max as spark_max, min as spark_min,
    dayofweek, month, to_date, abs as spark_abs,
    round as spark_round
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load raw data from Lakehouse (ingested by bronze notebook)
sensors_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('Files/data/grid_sensors.csv')
events_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('Files/data/power_events.csv')
renewable_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('Files/data/renewable_generation.csv')

print(f'Grid sensors: {sensors_df.count():,} rows')
print(f'Power events: {events_df.count():,} rows')
print(f'Renewable gen: {renewable_df.count():,} rows')

In [ ]:
# === Sensor features: aggregate per substation per day ===
sensors_df = sensors_df.withColumn('sensor_date', to_date('timestamp'))

sensor_daily = (
    sensors_df
    .groupBy('substation_id', 'region', 'sensor_date')
    .agg(
        avg('voltage_v').alias('avg_voltage'),
        stddev('voltage_v').alias('std_voltage'),
        spark_min('voltage_v').alias('min_voltage'),
        spark_max('voltage_v').alias('max_voltage'),
        avg('frequency_hz').alias('avg_frequency'),
        stddev('frequency_hz').alias('std_frequency'),
        avg('power_factor').alias('avg_power_factor'),
        spark_min('power_factor').alias('min_power_factor'),
        avg('load_mw').alias('avg_load'),
        spark_max('load_mw').alias('max_load'),
        avg('temperature_c').alias('avg_temp'),
        spark_max('temperature_c').alias('max_temp'),
        count('*').alias('reading_count'),
    )
    .withColumn('voltage_range', col('max_voltage') - col('min_voltage'))
    .withColumn('voltage_deviation', spark_abs(col('avg_voltage') - 230.0))  # deviation from nominal
    .withColumn('freq_deviation', spark_abs(col('avg_frequency') - 50.0))    # deviation from 50Hz
)

print(f'Sensor daily features: {sensor_daily.count():,} rows')

In [ ]:
# === Outage target: did this substation have an outage on this day? ===
outage_events = (
    events_df
    .filter(col('event_type').isin('outage', 'surge', 'sag'))
    .withColumn('event_date', to_date('timestamp'))
    .groupBy('substation_id', 'event_date')
    .agg(
        count('*').alias('event_count'),
        spark_sum('duration_sec').alias('total_event_duration'),
        spark_sum('affected_customers').alias('total_affected'),
        spark_max('severity').alias('max_severity'),
    )
    .withColumn('had_outage', lit(1))
)

print(f'Days with outage events: {outage_events.count():,}')

In [ ]:
# === Join sensor features with outage target ===
ml_features = (
    sensor_daily
    .join(
        outage_events.select('substation_id', col('event_date').alias('sensor_date'), 'had_outage', 'event_count', 'total_affected'),
        ['substation_id', 'sensor_date'],
        'left'
    )
    .withColumn('had_outage', when(col('had_outage').isNull(), 0).otherwise(1))
    .withColumn('day_of_week', dayofweek('sensor_date'))
    .withColumn('month', month('sensor_date'))
    .na.fill(0)
    .withColumn('feature_timestamp', current_timestamp())
)

ml_features.write.mode('overwrite').format('delta').saveAsTable('gold_ml_features')

outage_rate = ml_features.filter(col('had_outage') == 1).count() / ml_features.count() * 100
print(f'\nGold ML features written: {ml_features.count():,} rows')
print(f'Columns: {len(ml_features.columns)}')
print(f'Outage rate: {outage_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')